In [1]:
# ================================================================
# CLOUD-BASED STUDENT ASSIGNMENT SUBMISSION & FEEDBACK PORTAL
# GOOGLE COLAB - COMPLETE SINGLE-CELL APPLICATION (Gradio 6)
#
# Cloud concepts demonstrated:
#  - Client-server web app, publicly shared via Gradio tunnel
#  - Authentication (salted PBKDF2) + Role-Based Access Control
#  - Managed-DB pattern (SQLite, swappable for Firestore/Postgres)
#  - Object-storage pattern (ObjectStorage class, swappable for S3/GCS)
#  - Server-side deadline logic, versioning, idempotent uploads
#  - Notifications, audit log, deadline reminders, analytics, CSV export
#  - Secrets via environment variables (no hardcoded credentials)
#
# Demo accounts (dummy data) are seeded on first run.
# ================================================================

import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gradio>=6.0,<7.0"])

import os, re, uuid, shutil, sqlite3, hashlib, hmac, secrets, threading, html
from pathlib import Path
from datetime import datetime, timedelta, timezone

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import gradio as gr

# ================================================================
# 1. CONFIG (environment variables, safe defaults for demo)
# ================================================================
TZ = timezone(timedelta(hours=float(os.getenv("PORTAL_TZ_OFFSET", "5.5"))))   # IST default
TEACHER_CODE = os.getenv("TEACHER_INVITE_CODE", "TEACH-2026")                  # needed to register as teacher
DEMO_TEACHER_PW = os.getenv("DEMO_TEACHER_PW", "Teach@123")
DEMO_STUDENT_PW = os.getenv("DEMO_STUDENT_PW", "Stud@123")

APP_DIR = Path("/content/AssignmentPortal")
DB_PATH = APP_DIR / "database" / "portal.db"
BUCKET = APP_DIR / "object_storage" / "assignment-bucket"     # private "bucket"
EXPORT_DIR = APP_DIR / "exports"
for p in (DB_PATH.parent, BUCKET, EXPORT_DIR):
    p.mkdir(parents=True, exist_ok=True)

TEXT_EXT = {".txt", ".md", ".py", ".csv", ".java", ".c", ".cpp"}   # readable for similarity check


def now():
    return datetime.now(TZ)


def parse_dt(text):
    text = (text or "").strip()
    for f in ("%Y-%m-%d %H:%M", "%Y-%m-%d"):
        try:
            return datetime.strptime(text, f).replace(tzinfo=TZ)
        except ValueError:
            pass
    return None


def fmt(iso):
    try:
        return datetime.fromisoformat(iso).strftime("%d %b %Y, %I:%M %p")
    except Exception:
        return iso or "-"


def time_left(iso):
    d = datetime.fromisoformat(iso) - now()
    if d.total_seconds() < 0:
        return "overdue"
    return f"{d.days}d {d.seconds // 3600}h left"


# ================================================================
# 2. DATABASE (thread-safe SQLite)
# ================================================================
CONN = sqlite3.connect(DB_PATH, check_same_thread=False)
CONN.row_factory = sqlite3.Row
LOCK = threading.RLock()


def qa(sql, args=()):
    with LOCK:
        return CONN.execute(sql, args).fetchall()


def q1(sql, args=()):
    with LOCK:
        return CONN.execute(sql, args).fetchone()


def ex(sql, args=()):
    with LOCK:
        cur = CONN.execute(sql, args)
        CONN.commit()
        return cur.lastrowid


CONN.executescript("""
CREATE TABLE IF NOT EXISTS users(
  id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT UNIQUE NOT NULL, name TEXT NOT NULL,
  role TEXT NOT NULL CHECK(role IN ('student','teacher')), password_hash TEXT NOT NULL,
  salt TEXT NOT NULL, created_at TEXT NOT NULL);
CREATE TABLE IF NOT EXISTS courses(
  id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT NOT NULL,
  teacher_id INTEGER NOT NULL REFERENCES users(id), created_at TEXT NOT NULL);
CREATE TABLE IF NOT EXISTS assignments(
  id INTEGER PRIMARY KEY AUTOINCREMENT, course_id INTEGER NOT NULL REFERENCES courses(id),
  title TEXT NOT NULL, description TEXT, deadline TEXT NOT NULL, max_marks REAL NOT NULL,
  allowed_ext TEXT NOT NULL, max_mb REAL NOT NULL, allow_late INTEGER NOT NULL,
  allow_resubmit INTEGER NOT NULL, created_by INTEGER NOT NULL, created_at TEXT NOT NULL);
CREATE TABLE IF NOT EXISTS submissions(
  id INTEGER PRIMARY KEY AUTOINCREMENT, assignment_id INTEGER NOT NULL REFERENCES assignments(id),
  student_id INTEGER NOT NULL REFERENCES users(id), version INTEGER NOT NULL,
  file_name TEXT NOT NULL, storage_path TEXT NOT NULL, file_hash TEXT NOT NULL, size INTEGER NOT NULL,
  submitted_at TEXT NOT NULL, status TEXT NOT NULL, is_late INTEGER NOT NULL DEFAULT 0,
  marks REAL, feedback TEXT, graded_at TEXT, graded_by INTEGER);
CREATE INDEX IF NOT EXISTS idx_sub_lookup ON submissions(assignment_id, student_id, version);
CREATE TABLE IF NOT EXISTS notifications(
  id INTEGER PRIMARY KEY AUTOINCREMENT, user_id INTEGER NOT NULL, message TEXT NOT NULL,
  created_at TEXT NOT NULL, is_read INTEGER NOT NULL DEFAULT 0);
CREATE TABLE IF NOT EXISTS audit(
  id INTEGER PRIMARY KEY AUTOINCREMENT, user_id INTEGER, action TEXT, detail TEXT, created_at TEXT);
CREATE VIEW IF NOT EXISTS latest AS
  SELECT s.* FROM submissions s WHERE s.version =
  (SELECT MAX(version) FROM submissions WHERE assignment_id=s.assignment_id AND student_id=s.student_id);
""")


def notify(uid, msg):
    ex("INSERT INTO notifications(user_id,message,created_at) VALUES(?,?,?)", (uid, msg, now().isoformat()))


def audit(uid, action, detail=""):
    ex("INSERT INTO audit(user_id,action,detail,created_at) VALUES(?,?,?,?)", (uid, action, detail, now().isoformat()))


# ================================================================
# 3. OBJECT STORAGE (local stand-in for S3 / GCS / Firebase Storage)
#    Swap put/path/delete_prefix with boto3 or google-cloud-storage
#    calls to move to a real cloud bucket.
# ================================================================
class ObjectStorage:
    def __init__(self, root):
        self.root = Path(root)

    def put(self, key, src):
        dest = self.root / key
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dest)
        return key

    def path(self, key):
        return self.root / key

    def delete_prefix(self, prefix):
        shutil.rmtree(self.root / prefix, ignore_errors=True)


STORE = ObjectStorage(BUCKET)


# ================================================================
# 4. AUTH (PBKDF2 hashing, role-based guards)
# ================================================================
def hash_pw(pw, salt):
    return hashlib.pbkdf2_hmac("sha256", pw.encode(), bytes.fromhex(salt), 200_000).hex()


def create_user(username, name, pw, role):
    salt = secrets.token_hex(16)
    return ex("INSERT INTO users(username,name,role,password_hash,salt,created_at) VALUES(?,?,?,?,?,?)",
              (username, name, role, hash_pw(pw, salt), salt, now().isoformat()))


def register(username, name, password, role, code):
    username, name = (username or "").strip().lower(), (name or "").strip()
    if not re.fullmatch(r"[a-z0-9_]{3,20}", username):
        return "⚠️ Username: 3-20 chars, letters/numbers/underscore."
    if len(name) < 2 or len(password or "") < 6:
        return "⚠️ Enter your full name and a password of at least 6 characters."
    if role == "teacher" and not hmac.compare_digest(code or "", TEACHER_CODE):
        return "⛔ Invalid teacher invite code."
    try:
        uid = create_user(username, name, password, role)
        audit(uid, "REGISTER", role)
        return f"✅ Account created for **{name}** ({role}). You can sign in now."
    except sqlite3.IntegrityError:
        return "⚠️ Username already exists."


def login(username, password):
    u = q1("SELECT * FROM users WHERE username=?", ((username or "").strip().lower(),))
    if u and hmac.compare_digest(u["password_hash"], hash_pw(password or "", u["salt"])):
        audit(u["id"], "LOGIN")
        s = {"id": u["id"], "username": u["username"], "name": u["name"], "role": u["role"]}
        return s, f"🟢 Signed in as **{u['name']}** ({u['role']})."
    return {}, "❌ Incorrect username or password."


def logout(sess):
    if sess:
        audit(sess.get("id"), "LOGOUT")
    return {}, "👋 Signed out."


def is_role(sess, role):
    return bool(sess) and sess.get("role") == role


DENIED = "⛔ Access denied - this action needs a signed-in {} account."


# ================================================================
# 5. SEED DUMMY DATA
# ================================================================
def seed():
    if q1("SELECT 1 FROM users LIMIT 1"):
        return
    t = create_user("prof_rao", "Prof. Rao", DEMO_TEACHER_PW, "teacher")
    create_user("asha", "Asha Verma", DEMO_STUDENT_PW, "student")
    create_user("ravi", "Ravi Kumar", DEMO_STUDENT_PW, "student")
    c = ex("INSERT INTO courses(name,teacher_id,created_at) VALUES(?,?,?)",
           ("CS101 - Cloud Computing", t, now().isoformat()))
    ex("""INSERT INTO assignments(course_id,title,description,deadline,max_marks,allowed_ext,max_mb,
          allow_late,allow_resubmit,created_by,created_at) VALUES(?,?,?,?,?,?,?,?,?,?,?)""",
       (c, "Assignment 1: Cloud Service Models",
        "Compare IaaS, PaaS and SaaS with one real example each.",
        (now() + timedelta(days=3)).replace(hour=23, minute=59, second=0, microsecond=0).isoformat(),
        20, "pdf,docx,txt,png,jpg,zip", 5, 1, 1, t, now().isoformat()))


seed()


# ================================================================
# 6. ASSIGNMENT MANAGEMENT (teacher)
# ================================================================
def teacher_courses(sess):
    return qa("SELECT * FROM courses WHERE teacher_id=?", (sess["id"],))


def create_course(sess, name):
    if not is_role(sess, "teacher"):
        return DENIED.format("teacher")
    if len((name or "").strip()) < 3:
        return "⚠️ Course name too short."
    ex("INSERT INTO courses(name,teacher_id,created_at) VALUES(?,?,?)", (name.strip(), sess["id"], now().isoformat()))
    audit(sess["id"], "CREATE_COURSE", name)
    return f"✅ Course **{name.strip()}** created."


def create_assignment(sess, course_id, title, desc, deadline, max_marks, exts, max_mb, allow_late, allow_resub):
    if not is_role(sess, "teacher"):
        return DENIED.format("teacher")
    if not course_id or not q1("SELECT 1 FROM courses WHERE id=? AND teacher_id=?", (course_id, sess["id"])):
        return "⛔ Choose one of your own courses."
    dl = parse_dt(deadline)
    if not (title or "").strip() or dl is None:
        return "⚠️ Title is required and deadline must be `YYYY-MM-DD HH:MM`."
    try:
        max_marks, max_mb = float(max_marks), float(max_mb)
        assert max_marks > 0 and 0 < max_mb <= 50
    except Exception:
        return "⚠️ Max marks must be > 0 and max size between 0 and 50 MB."
    ext_list = sorted({e.strip().lower().lstrip(".") for e in (exts or "").split(",") if e.strip()})
    if not ext_list:
        return "⚠️ Provide at least one allowed extension."
    ex("""INSERT INTO assignments(course_id,title,description,deadline,max_marks,allowed_ext,max_mb,
          allow_late,allow_resubmit,created_by,created_at) VALUES(?,?,?,?,?,?,?,?,?,?,?)""",
       (course_id, title.strip(), desc or "", dl.isoformat(), max_marks, ",".join(ext_list), max_mb,
        int(allow_late), int(allow_resub), sess["id"], now().isoformat()))
    for s in qa("SELECT id FROM users WHERE role='student'"):
        notify(s["id"], f"📢 New assignment: {title.strip()} (due {fmt(dl.isoformat())})")
    audit(sess["id"], "CREATE_ASSIGNMENT", title)
    return f"✅ Assignment **{title.strip()}** created. Students have been notified."


def delete_assignment(sess, aid):
    if not is_role(sess, "teacher"):
        return DENIED.format("teacher")
    a = owned_assignment(sess, aid)
    if not a:
        return "⛔ Select one of your own assignments."
    ex("DELETE FROM submissions WHERE assignment_id=?", (aid,))
    ex("DELETE FROM assignments WHERE id=?", (aid,))
    STORE.delete_prefix(f"assignments/assignment_{aid:03d}")
    audit(sess["id"], "DELETE_ASSIGNMENT", str(aid))
    return f"🗑️ Assignment #{aid} and its files were deleted."


def owned_assignment(sess, aid):
    if not aid or not is_role(sess, "teacher"):
        return None
    return q1("""SELECT a.*, c.name AS course FROM assignments a JOIN courses c ON c.id=a.course_id
                 WHERE a.id=? AND c.teacher_id=?""", (aid, sess["id"]))


def teacher_assignments(sess):
    return qa("""SELECT a.*, c.name AS course FROM assignments a JOIN courses c ON c.id=a.course_id
                 WHERE c.teacher_id=? ORDER BY a.deadline""", (sess["id"],))


# ================================================================
# 7. SUBMISSION (student) - validation, deadline, versioning
# ================================================================
def submit_assignment(sess, aid, file):
    if not is_role(sess, "student"):
        return DENIED.format("student")
    a = q1("SELECT a.*, c.teacher_id FROM assignments a JOIN courses c ON c.id=a.course_id WHERE a.id=?", (aid,))
    if not a:
        return "⚠️ Select an assignment."
    if not file:
        return "⚠️ Choose a file to upload."
    name = re.sub(r"[^\w.\-]", "_", Path(file).name)
    ext = Path(name).suffix.lower().lstrip(".")
    if ext not in a["allowed_ext"].split(","):
        return f"❌ File type `.{ext}` not allowed. Allowed: {a['allowed_ext']}"
    size = os.path.getsize(file)
    if size > a["max_mb"] * 1024 * 1024:
        return f"❌ File is {size / 1048576:.1f} MB; limit is {a['max_mb']:g} MB."

    # Server-side clock only - client time is never trusted.
    late = now() > datetime.fromisoformat(a["deadline"])
    if late and not a["allow_late"]:
        return "⛔ Deadline passed and late submissions are disabled for this assignment."

    prev = q1("SELECT * FROM latest WHERE assignment_id=? AND student_id=?", (aid, sess["id"]))
    if prev:
        if not a["allow_resubmit"]:
            return "⛔ Resubmission is not allowed for this assignment."
        if prev["status"] == "GRADED":
            return "⛔ This submission is already graded and can no longer be replaced."
    digest = hashlib.sha256(Path(file).read_bytes()).hexdigest()
    if prev and prev["file_hash"] == digest:      # idempotency: same file twice = no new version
        return f"ℹ️ Identical file already submitted as version {prev['version']}. No duplicate created."

    version = (prev["version"] + 1) if prev else 1
    key = f"assignments/assignment_{aid:03d}/student_{sess['id']:03d}/v{version}_{uuid.uuid4().hex[:8]}.{ext}"
    try:
        STORE.put(key, file)
    except Exception as e:
        audit(sess["id"], "UPLOAD_FAILED", str(e))
        return "❌ Cloud storage error - nothing was saved. Please retry."
    status = "LATE" if late else "SUBMITTED"
    sid = ex("""INSERT INTO submissions(assignment_id,student_id,version,file_name,storage_path,file_hash,size,
                submitted_at,status,is_late) VALUES(?,?,?,?,?,?,?,?,?,?)""",
             (aid, sess["id"], version, name, key, digest, size, now().isoformat(), status, int(late)))
    notify(a["teacher_id"], f"📥 {sess['name']} submitted '{a['title']}' (v{version}, {status})")
    audit(sess["id"], "SUBMIT", f"assignment={aid} v{version}")
    return (f"### ✅ Submission received\n| Field | Value |\n|---|---|\n| Receipt ID | **#{sid}** |\n"
            f"| File | {name} ({size / 1024:.1f} KB) |\n| Version | {version} |\n| Status | **{status}** |\n"
            f"| Time (server) | {fmt(now().isoformat())} |")


def my_submissions(sess):
    if not is_role(sess, "student"):
        return pd.DataFrame()
    rows = qa("""SELECT s.id AS ID, a.title AS Assignment, s.version AS Version, s.file_name AS File,
                 s.submitted_at, s.status AS Status, s.marks, a.max_marks, s.feedback AS Feedback
                 FROM submissions s JOIN assignments a ON a.id=s.assignment_id
                 WHERE s.student_id=? ORDER BY s.id DESC""", (sess["id"],))
    return pd.DataFrame([{
        "ID": r["ID"], "Assignment": r["Assignment"], "Ver": r["Version"], "File": r["File"],
        "Submitted": fmt(r["submitted_at"]), "Status": r["Status"],
        "Marks": f"{r['marks']:g}/{r['max_marks']:g}" if r["marks"] is not None else "-",
        "Feedback": r["Feedback"] or "-"} for r in rows])


def download_submission(sess, sub_id):
    if not sess:
        return None
    s = q1("""SELECT s.*, c.teacher_id FROM submissions s JOIN assignments a ON a.id=s.assignment_id
              JOIN courses c ON c.id=a.course_id WHERE s.id=?""", (sub_id,))
    if not s:
        return None
    owner = is_role(sess, "student") and s["student_id"] == sess["id"]
    reviewer = is_role(sess, "teacher") and s["teacher_id"] == sess["id"]
    if not (owner or reviewer):                       # RBAC / ownership check
        audit(sess["id"], "DENIED_DOWNLOAD", str(sub_id))
        return None
    dest = EXPORT_DIR / f"sub{s['id']}_v{s['version']}_{s['file_name']}"
    shutil.copy2(STORE.path(s["storage_path"]), dest)
    return str(dest)


# ================================================================
# 8. GRADING & FEEDBACK (teacher)
# ================================================================
def teacher_submissions(sess, aid):
    a = owned_assignment(sess, aid)
    if not a:
        return pd.DataFrame(), gr.Dropdown(choices=[], value=None)
    rows = qa("""SELECT s.*, u.name FROM submissions s JOIN users u ON u.id=s.student_id
                 WHERE s.assignment_id=? ORDER BY u.name, s.version DESC""", (aid,))
    df = pd.DataFrame([{
        "ID": r["id"], "Student": r["name"], "Ver": r["version"], "File": r["file_name"],
        "Submitted": fmt(r["submitted_at"]), "Status": r["status"], "Late": "Yes" if r["is_late"] else "No",
        "Marks": f"{r['marks']:g}/{a['max_marks']:g}" if r["marks"] is not None else "-",
        "Feedback": r["feedback"] or "-"} for r in rows])
    choices = [(f"#{r['id']} - {r['name']} v{r['version']} ({r['status']})", r["id"]) for r in rows]
    return df, gr.Dropdown(choices=choices, value=None)


def grade_submission(sess, sub_id, marks, feedback):
    if not is_role(sess, "teacher"):
        return DENIED.format("teacher")
    s = q1("""SELECT s.*, a.max_marks, a.title, c.teacher_id FROM submissions s
              JOIN assignments a ON a.id=s.assignment_id JOIN courses c ON c.id=a.course_id WHERE s.id=?""", (sub_id,))
    if not s or s["teacher_id"] != sess["id"]:
        return "⛔ You can only grade submissions from your own courses."
    try:
        marks = float(marks)
    except Exception:
        return "⚠️ Enter numeric marks."
    if marks < 0 or marks > s["max_marks"]:
        return f"❌ Marks must be between 0 and {s['max_marks']:g}."
    if not (feedback or "").strip():
        return "⚠️ Please write some feedback."
    ex("UPDATE submissions SET marks=?, feedback=?, status='GRADED', graded_at=?, graded_by=? WHERE id=?",
       (marks, feedback.strip(), now().isoformat(), sess["id"], sub_id))
    notify(s["student_id"], f"📝 '{s['title']}' graded: {marks:g}/{s['max_marks']:g}. Feedback is available.")
    audit(sess["id"], "GRADE", f"submission={sub_id}")
    return f"✅ Graded submission #{sub_id}: **{marks:g}/{s['max_marks']:g}**. Student notified."


def similarity_check(sess, aid):
    """Pairwise TF-IDF cosine similarity on text-based submissions (starter plagiarism check)."""
    if not owned_assignment(sess, aid):
        return pd.DataFrame()
    names, texts = [], []
    for r in qa("SELECT l.*, u.name FROM latest l JOIN users u ON u.id=l.student_id WHERE l.assignment_id=?", (aid,)):
        if Path(r["file_name"]).suffix.lower() in TEXT_EXT:
            names.append(r["name"])
            texts.append(STORE.path(r["storage_path"]).read_text(errors="ignore"))
    if len(texts) < 2:
        return pd.DataFrame([{"Info": "Need at least 2 text-based submissions (.txt/.md/.py/.csv)."}])
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    M = cosine_similarity(TfidfVectorizer(stop_words="english").fit_transform(texts))
    pairs = [{"Student A": names[i], "Student B": names[j], "Similarity %": round(M[i][j] * 100, 1)}
             for i in range(len(names)) for j in range(i + 1, len(names))]
    return pd.DataFrame(pairs).sort_values("Similarity %", ascending=False)


def export_gradebook(sess, aid):
    a = owned_assignment(sess, aid)
    if not a:
        return None
    rows = qa("""SELECT u.name, u.username, l.version, l.submitted_at, l.status, l.marks, l.feedback
                 FROM latest l JOIN users u ON u.id=l.student_id WHERE l.assignment_id=?""", (aid,))
    df = pd.DataFrame([dict(r) for r in rows])
    path = EXPORT_DIR / f"gradebook_assignment_{aid}.csv"
    df.to_csv(path, index=False)
    return str(path)


# ================================================================
# 9. DASHBOARDS
# ================================================================
def kpi(label, value, cls=""):
    return f'<div class="kpi {cls}"><span>{label}</span><b>{value}</b></div>'


def empty_state(icon, title, text):
    return f'<div class="empty"><div>{icon}</div><h2>{title}</h2><p>{text}</p></div>'


def student_dashboard(sess):
    sid = sess["id"]
    latest = {r["assignment_id"]: r for r in qa("SELECT * FROM latest WHERE student_id=?", (sid,))}
    counts = {"Not submitted": 0, "Submitted": 0, "Late": 0, "Graded": 0}
    late_n, upcoming, feedback = 0, [], []
    assignments = qa("SELECT a.*, c.name AS course FROM assignments a JOIN courses c ON c.id=a.course_id ORDER BY a.deadline")
    for a in assignments:
        s = latest.get(a["id"])
        if not s:
            counts["Not submitted"] += 1
            upcoming.append(a)
            continue
        late_n += s["is_late"]
        if s["status"] == "GRADED":
            counts["Graded"] += 1
            feedback.append((a, s))
        elif s["status"] == "LATE":
            counts["Late"] += 1
        else:
            counts["Submitted"] += 1
    unread = q1("SELECT COUNT(*) n FROM notifications WHERE user_id=? AND is_read=0", (sid,))["n"]
    up = "".join(
        f'<li class="{"warn" if (datetime.fromisoformat(a["deadline"]) - now()).total_seconds() < 172800 else ""}">'
        f'<b>{html.escape(a["title"])}</b><span>{html.escape(a["course"])} · due {fmt(a["deadline"])} · '
        f'<em>{time_left(a["deadline"])}</em></span></li>' for a in upcoming) or "<li>🎉 Nothing pending.</li>"
    fb = "".join(
        f'<li><b>{html.escape(a["title"])} - {s["marks"]:g}/{a["max_marks"]:g}</b>'
        f'<span>{html.escape(s["feedback"] or "")}</span></li>'
        for a, s in sorted(feedback, key=lambda x: x[1]["graded_at"], reverse=True)[:3]) or "<li>No feedback yet.</li>"
    dash = f"""<div class="dash"><div class="micro">STUDENT DASHBOARD</div><h1>Welcome, {html.escape(sess['name'])} ✦</h1>
    <div class="grid">{kpi('Total assignments', len(assignments))}{kpi('Pending', counts['Not submitted'])}
    {kpi('Submitted', counts['Submitted'] + counts['Late'] + counts['Graded'])}{kpi('Late', late_n)}
    {kpi('Graded', counts['Graded'])}{kpi('Unread alerts', unread, 'accent')}</div>
    <div class="cols"><div class="card"><h3>⏰ Upcoming deadlines &amp; reminders</h3><ul>{up}</ul></div>
    <div class="card"><h3>💬 Recent feedback</h3><ul>{fb}</ul></div></div></div>"""
    return dash, counts


def teacher_dashboard(sess):
    aids = [a["id"] for a in teacher_assignments(sess)]
    n_students = q1("SELECT COUNT(*) n FROM users WHERE role='student'")["n"]
    subs = []
    if aids:
        ph = ",".join("?" * len(aids))
        subs = qa(f"""SELECT l.*, u.name, a.title FROM latest l JOIN users u ON u.id=l.student_id
                      JOIN assignments a ON a.id=l.assignment_id WHERE l.assignment_id IN ({ph})
                      ORDER BY l.submitted_at DESC""", aids)
    graded = sum(1 for s in subs if s["status"] == "GRADED")
    pending = sum(1 for s in subs if s["status"] in ("SUBMITTED", "LATE"))
    late = sum(1 for s in subs if s["is_late"])
    missing = max(0, n_students * len(aids) - len(subs))
    counts = {"Not submitted": missing, "Submitted": sum(1 for s in subs if s["status"] == "SUBMITTED"),
              "Late": sum(1 for s in subs if s["status"] == "LATE"), "Graded": graded}
    recent = "".join(f'<li><b>{html.escape(s["name"])}</b><span>{html.escape(s["title"])} · v{s["version"]} · '
                     f'{fmt(s["submitted_at"])} · {s["status"]}</span></li>' for s in subs[:5]) or "<li>No uploads yet.</li>"
    upcoming = "".join(f'<li><b>{html.escape(a["title"])}</b><span>{fmt(a["deadline"])} · <em>{time_left(a["deadline"])}</em></span></li>'
                       for a in teacher_assignments(sess) if datetime.fromisoformat(a["deadline"]) > now()) or "<li>None.</li>"
    unread = q1("SELECT COUNT(*) n FROM notifications WHERE user_id=? AND is_read=0", (sess["id"],))["n"]
    dash = f"""<div class="dash"><div class="micro">TEACHER DASHBOARD</div><h1>Welcome, {html.escape(sess['name'])} ✦</h1>
    <div class="grid">{kpi('Assignments', len(aids))}{kpi('Students', n_students)}{kpi('Submissions', len(subs))}
    {kpi('Pending reviews', pending, 'accent')}{kpi('Late submissions', late)}{kpi('Graded', graded)}</div>
    <div class="cols"><div class="card"><h3>📥 Recent uploads</h3><ul>{recent}</ul></div>
    <div class="card"><h3>⏰ Upcoming deadlines</h3><ul>{upcoming}</ul>
    <p class="muted">🔔 {unread} unread notifications</p></div></div></div>"""
    return dash, counts


def status_chart(counts, title):
    fig, ax = plt.subplots(figsize=(6.5, 3.3))
    fig.patch.set_alpha(0)
    if counts is None or sum(counts.values()) == 0:
        ax.text(0.5, 0.5, "No data yet", ha="center", va="center", fontsize=13, color="#64748B")
        ax.axis("off")
        return fig
    colors = ["#94A3B8", "#6C63FF", "#F59E0B", "#22C55E"]
    bars = ax.bar(list(counts), list(counts.values()), color=colors)
    ax.bar_label(bars, padding=2)
    ax.set_title(title, fontsize=11, color="#334155")
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    fig.tight_layout()
    return fig


def notifications_df(sess):
    if not sess:
        return pd.DataFrame()
    rows = qa("SELECT * FROM notifications WHERE user_id=? ORDER BY id DESC LIMIT 25", (sess["id"],))
    return pd.DataFrame([{"When": fmt(r["created_at"]), "Message": r["message"],
                          "Read": "✓" if r["is_read"] else "●"} for r in rows])


def mark_read(sess):
    if sess:
        ex("UPDATE notifications SET is_read=1 WHERE user_id=?", (sess["id"],))
    return notifications_df(sess)


def teacher_assignments_df(sess):
    if not is_role(sess, "teacher"):
        return pd.DataFrame()
    return pd.DataFrame([{"ID": a["id"], "Course": a["course"], "Title": a["title"], "Deadline": fmt(a["deadline"]),
                          "Max": a["max_marks"], "Types": a["allowed_ext"], "MB": a["max_mb"],
                          "Late OK": bool(a["allow_late"]), "Resubmit": bool(a["allow_resubmit"])}
                         for a in teacher_assignments(sess)])


def refresh_all(sess):
    """Rebuild every role-dependent component after login/logout/any change."""
    no_dd = gr.Dropdown(choices=[], value=None)
    if not sess:
        return (empty_state("🔐", "Your Cloud Portal", "Sign in from the Account tab to continue."),
                status_chart(None, ""), pd.DataFrame(), no_dd, no_dd, no_dd, no_dd, pd.DataFrame(), pd.DataFrame())
    if sess["role"] == "student":
        dash, counts = student_dashboard(sess)
        a_choices = [(f"#{a['id']} · {a['title']} ({a['course']}) · due {fmt(a['deadline'])}", a["id"])
                     for a in qa("SELECT a.*, c.name AS course FROM assignments a JOIN courses c ON c.id=a.course_id ORDER BY a.deadline")]
        my = my_submissions(sess)
        sub_choices = [(f"#{r['ID']} · {r['Assignment']} v{r['Ver']}", r["ID"]) for _, r in my.iterrows()]
        return (dash, status_chart(counts, "My assignment status"), notifications_df(sess),
                gr.Dropdown(choices=a_choices, value=None), gr.Dropdown(choices=sub_choices, value=None),
                no_dd, no_dd, pd.DataFrame(), my)
    dash, counts = teacher_dashboard(sess)
    course_choices = [(c["name"], c["id"]) for c in teacher_courses(sess)]
    a_choices = [(f"#{a['id']} · {a['title']} ({a['course']})", a["id"]) for a in teacher_assignments(sess)]
    return (dash, status_chart(counts, "Class submission status"), notifications_df(sess), no_dd, no_dd,
            gr.Dropdown(choices=course_choices, value=None), gr.Dropdown(choices=a_choices, value=None),
            teacher_assignments_df(sess), pd.DataFrame())


# ================================================================
# 10. CSS
# ================================================================
CSS = r"""
body{background:#F4F6FA!important;font-family:Inter,system-ui,sans-serif!important}
.gradio-container{max-width:1300px!important;margin:auto!important}
footer{display:none!important}
.app-header{padding:34px;margin-bottom:16px;border-radius:24px;color:#fff;
 background:linear-gradient(135deg,#0E1330,#252B68 55%,#635BFF);box-shadow:0 20px 50px rgba(37,43,104,.2)}
.app-header h1{color:#fff!important;margin:0;font-size:34px;letter-spacing:-1px}
.app-header p{margin:6px 0 0;color:rgba(255,255,255,.75);font-size:13px}
.micro{color:#8A92A6;font-size:10px;font-weight:800;letter-spacing:1.6px}
.dash h1{margin:4px 0 16px;color:#111827;font-size:28px}
.grid{display:grid;grid-template-columns:repeat(6,1fr);gap:12px;margin-bottom:14px}
.kpi{background:#fff;border:1px solid #E6E9F0;border-radius:16px;padding:16px}
.kpi span{display:block;color:#8A92A6;font-size:10px;font-weight:700}
.kpi b{font-size:26px;color:#111827}
.kpi.accent{background:linear-gradient(145deg,#171C43,#373C89)}
.kpi.accent span{color:rgba(255,255,255,.65)}.kpi.accent b{color:#fff}
.cols{display:grid;grid-template-columns:1fr 1fr;gap:14px}
.card{background:#fff;border:1px solid #E6E9F0;border-radius:18px;padding:18px}
.card h3{margin:0 0 8px;color:#111827;font-size:15px}
.card ul{list-style:none;padding:0;margin:0}
.card li{padding:9px 0;border-bottom:1px solid #F0F1F5;font-size:12px;color:#374151}
.card li span{display:block;color:#7B8497;margin-top:2px}
.card li.warn{background:#FFF7ED;border-radius:8px;padding:9px}
.muted{color:#7B8497;font-size:11px}
.empty{text-align:center;padding:70px 20px;background:#fff;border:1px solid #E6E9F0;border-radius:22px}
.empty div{font-size:40px}.empty h2{color:#111827}.empty p{color:#7B8497}
@media(max-width:900px){.grid{grid-template-columns:repeat(2,1fr)}.cols{grid-template-columns:1fr}}
"""

# ================================================================
# 11. GRADIO APP
# ================================================================
with gr.Blocks(title="Cloud Assignment Portal") as app:
    sess = gr.State({})

    gr.HTML("""<div class="app-header"><h1>✦ Cloud Assignment Portal</h1>
    <p>Submission &amp; Feedback Platform &nbsp;•&nbsp; Role-Based Access &nbsp;•&nbsp; Object Storage &nbsp;•&nbsp; Real-Time Tracking</p></div>""")

    # ---------------- Dashboard ----------------
    with gr.Tab("◈ Dashboard"):
        refresh_btn = gr.Button("↻ Refresh", variant="primary")
        dash_html = gr.HTML(empty_state("🔐", "Your Cloud Portal", "Sign in from the Account tab to continue."))
        dash_plot = gr.Plot(value=status_chart(None, ""), label="Status overview")

    # ---------------- Account ----------------
    with gr.Tab("⌁ Account"):
        gr.Markdown("**Demo accounts (dummy data):** teacher `prof_rao` / `Teach@123` · students `asha`, `ravi` / `Stud@123`. "
                    "Teacher registration needs the invite code `TEACH-2026` (set via `TEACHER_INVITE_CODE`).")
        with gr.Row():
            with gr.Column():
                gr.Markdown("### Register")
                r_user, r_name = gr.Textbox(label="Username"), gr.Textbox(label="Full name")
                r_pw = gr.Textbox(label="Password", type="password")
                r_role = gr.Radio(["student", "teacher"], value="student", label="Role")
                r_code = gr.Textbox(label="Teacher invite code (teachers only)", type="password")
                r_btn = gr.Button("Create account", variant="primary")
                r_out = gr.Markdown()
            with gr.Column():
                gr.Markdown("### Sign in")
                l_user = gr.Textbox(label="Username")
                l_pw = gr.Textbox(label="Password", type="password")
                l_btn = gr.Button("Sign in", variant="primary")
                lo_btn = gr.Button("Sign out")
                l_out = gr.Markdown()

    # ---------------- Student ----------------
    with gr.Tab("🎓 Student"):
        gr.Markdown("## Submit an assignment")
        s_assign = gr.Dropdown(label="Assignment", choices=[])
        s_file = gr.File(label="Upload file (type / size validated on the server)", type="filepath")
        s_btn = gr.Button("⬆ Submit / Resubmit", variant="primary")
        s_out = gr.Markdown()
        gr.Markdown("## My submissions, marks & feedback")
        s_table = gr.Dataframe(interactive=False, wrap=True)
        with gr.Row():
            s_dl_dd = gr.Dropdown(label="Download a previous submission", choices=[])
            s_dl_btn = gr.Button("⬇ Download")
        s_dl_file = gr.File(label="Your file")

    # ---------------- Teacher ----------------
    with gr.Tab("👩‍🏫 Teacher"):
        gr.Markdown("## Courses & assignments")
        with gr.Row():
            c_name = gr.Textbox(label="New course name")
            c_btn = gr.Button("Create course")
        c_out = gr.Markdown()
        with gr.Row():
            t_course = gr.Dropdown(label="Course", choices=[])
            t_title = gr.Textbox(label="Title")
        t_desc = gr.Textbox(label="Description", lines=2)
        with gr.Row():
            t_deadline = gr.Textbox(label="Deadline (YYYY-MM-DD HH:MM, IST)",
                                    value=(now() + timedelta(days=7)).strftime("%Y-%m-%d 23:59"))
            t_marks = gr.Number(label="Max marks", value=100)
            t_exts = gr.Textbox(label="Allowed extensions", value="pdf,docx,txt,png,jpg,zip")
            t_mb = gr.Number(label="Max size (MB)", value=5)
        with gr.Row():
            t_late = gr.Checkbox(label="Allow late submissions", value=True)
            t_resub = gr.Checkbox(label="Allow resubmission", value=True)
        t_create = gr.Button("＋ Create assignment", variant="primary")
        t_out = gr.Markdown()
        t_table = gr.Dataframe(interactive=False, wrap=True, label="My assignments")

        gr.Markdown("## Review, grade & give feedback")
        with gr.Row():
            t_assign = gr.Dropdown(label="Assignment", choices=[])
            t_load = gr.Button("Load submissions")
            t_del = gr.Button("🗑 Delete assignment")
        t_del_out = gr.Markdown()
        t_subs = gr.Dataframe(interactive=False, wrap=True, label="Submissions (all versions)")
        with gr.Row():
            g_sub = gr.Dropdown(label="Submission", choices=[])
            g_marks = gr.Number(label="Marks")
        g_fb = gr.Textbox(label="Feedback", lines=3)
        with gr.Row():
            g_btn = gr.Button("✔ Submit grade", variant="primary")
            g_dl = gr.Button("⬇ Open file")
        g_out = gr.Markdown()
        g_file = gr.File(label="Submission file")
        with gr.Row():
            sim_btn = gr.Button("🔍 Similarity check (text files)")
            exp_btn = gr.Button("↓ Export gradebook CSV")
        sim_table = gr.Dataframe(interactive=False)
        exp_file = gr.File(label="Gradebook")

    # ---------------- Notifications ----------------
    with gr.Tab("🔔 Notifications"):
        n_btn = gr.Button("Mark all as read")
        n_table = gr.Dataframe(interactive=False, wrap=True)

    # ---------------- Wiring ----------------
    REFRESH = [dash_html, dash_plot, n_table, s_assign, s_dl_dd, t_course, t_assign, t_table, s_table]

    refresh_btn.click(refresh_all, [sess], REFRESH)
    r_btn.click(register, [r_user, r_name, r_pw, r_role, r_code], [r_out])
    l_btn.click(login, [l_user, l_pw], [sess, l_out]).then(refresh_all, [sess], REFRESH)
    lo_btn.click(logout, [sess], [sess, l_out]).then(refresh_all, [sess], REFRESH)

    s_btn.click(submit_assignment, [sess, s_assign, s_file], [s_out]).then(refresh_all, [sess], REFRESH)
    s_dl_btn.click(download_submission, [sess, s_dl_dd], [s_dl_file])

    c_btn.click(create_course, [sess, c_name], [c_out]).then(refresh_all, [sess], REFRESH)
    t_create.click(create_assignment,
                   [sess, t_course, t_title, t_desc, t_deadline, t_marks, t_exts, t_mb, t_late, t_resub],
                   [t_out]).then(refresh_all, [sess], REFRESH)
    t_del.click(delete_assignment, [sess, t_assign], [t_del_out]).then(refresh_all, [sess], REFRESH)
    t_load.click(teacher_submissions, [sess, t_assign], [t_subs, g_sub])
    g_btn.click(grade_submission, [sess, g_sub, g_marks, g_fb], [g_out]) \
         .then(teacher_submissions, [sess, t_assign], [t_subs, g_sub]) \
         .then(refresh_all, [sess], REFRESH)
    g_dl.click(download_submission, [sess, g_sub], [g_file])
    sim_btn.click(similarity_check, [sess, t_assign], [sim_table])
    exp_btn.click(export_gradebook, [sess, t_assign], [exp_file])
    n_btn.click(mark_read, [sess], [n_table])

print("=" * 70)
print("✦ CLOUD ASSIGNMENT PORTAL READY")
print("  Teacher : prof_rao / Teach@123   Students: asha, ravi / Stud@123")
print("=" * 70)

app.launch(
    share=True,
    debug=False,
    css=CSS,
    theme=gr.themes.Soft(primary_hue="indigo", secondary_hue="violet", neutral_hue="slate"),
)

✦ CLOUD ASSIGNMENT PORTAL READY
  Teacher : prof_rao / Teach@123   Students: asha, ravi / Stud@123
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://69ea99db46e6b302d2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
